In [2]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## SQLの分類

SQLは用途に応じて、`DDL`、`DML`、`DCL`などに分類される。

| 分類 | 正式名称 | 意味 | 主な命令 | 用途 |
|---|---|---|---|---|
| `DDL` | `Data Definition Language` | データ定義語 | `CREATE`, `DROP`, `ALTER`,  `TRUNCATE` | テーブル構造の`作成・変更・削除、全データ削除` |
| `DML` | `Data Manipulation Language` | データ操作語 | `INSERT`, `UPDATE`, `DELETE`, `SELECT` | データの`照会・追加・修正・削除` |
| `DCL` | `Data Control Language` | データ制御語 | `GRANT`, `REVOKE` | 権限の`付与・取消` |

#### ※ `DELETE`はデータを行単位で削除するため、`DML`に分類される。
#### ※ `TRUNCATE`はテーブル内の全データを削除する点では`DELETE`に似ているが、MySQLでは`DDL`に分類される。

# 1. product テーブルを作成するSQLクエリ
> DDL : CREATE, DROP, ALTER

- `AUTO_INCREMENT`は、新しい行が追加されるたびに、数値を`自動的に1ずつ増やして保存する属性`である。

- 主に`主キー（Primary Key）`に使用し、各行に`一意の番号`を自動的に付与する。

In [4]:
%%sql

CREATE DATABASE test;

++
||
++
++

In [7]:
%%sql

CREATE TABLE product (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(30),
    price INT
);

++
||
++
++

# 2. データの照会・入力・修正・削除を行うSQLクエリ
> DML : INSERT, UPDATE, DELETE, SELECT

#### -- 追加

In [8]:
%%sql

insert into product(name, price)
values ('キーボード', 50000);

insert into product(name, price)
values ('マウス', 30000);

++
||
++
++

#### -- 検索

In [3]:
%%sql

select * from product;

id,name,price
1,キーボード,50000
2,マウス,30000


#### -- 修正

In [ ]:
%%sql

UPDATE product
SET price = 35000
WHERE id = 2;

++
||
++
++

#### -- 再検索

In [5]:
%%sql

SELECT * FROM product;

id,name,price
1,キーボード,50000
2,マウス,35000


#### -- 削除

In [6]:
%%sql

DELETE FROM product
WHERE id = 1;

++
||
++
++

#### -- 再検索

In [7]:
%%sql

SELECT * FROM product;

id,name,price
2,マウス,35000


#### -- TRUNCATE文を使用して、テーブル内の全データを削除する

In [9]:
%%sql

TRUNCATE product;

++
||
++
++

#### -- 最終検索

In [10]:
%%sql

SELECT * FROM product;

id,name,price


## ※ `DELETE` vs `TRUNCATE`

| `DELETE` | `TRUNCATE` |
|---|---|
| 条件を指定して、`特定の行を削除`できる | テーブル内の`すべての行を削除`する |
| `WHERE`句を使用できる | `WHERE`句を使用できない |
| `AUTO_INCREMENT`の値は基本的に維持される | `AUTO_INCREMENT`の値が初期化される |
| `ROLLBACK`(以前の状態に戻す)できる | 基本的に`ROLLBACK`できない |

- 大量のデータを削除する場合、`DELETE`文ではトランザクションログが多く記録されるため、データベースの性能に影響を与えることがある。  
- すべてのデータを高速に削除し、ロールバックを考慮しない場合は、`TRUNCATE`を使用する。

## 3. `NULL` vs `NOT NULL`

- `NULL`は、`0`でも空文字列 `""` でもない。  
- `値がまだ存在しない`、または`不明である状態`を表す。

| 区分 | 意味 |
|---|---|
| `NULL` | 値がない状態を許可する |
| `NOT NULL` | 必ず値を入力しなければならない |
| `PRIMARY KEY` | 各行を一意に識別するキー。自動的に`NOT NULL`になる |

In [ ]:
%%sql

CREATE TABLE product2 (
    id INT AUTO_INCREMENT PRIMARY KEY, -- PRIMARY KEYは自動的にNOT NULLになる
    name VARCHAR(30),                  -- NOT NULLを指定していないため、NULLを許可する
    price INT NOT NULL                 -- priceはNULLを許可しない
);

++
||
++
++

#### product1

In [12]:
%%sql

INSERT INTO product(name)
VALUES('モニター');
INSERT INTO product(name, price)
VALUES('カメラ', NULL)

++
||
++
++

In [13]:
%%sql

SELECT * FROM product

id,name,price
1,モニター,None
2,カメラ,None


#### ※ SQLの`NULL`は、Python環境では`None`として表示される場合がある。

- これはデータが`None`として保存されたという意味ではなく、  
- SQLの`NULL`がPython上で`None`に変換されて表示されているだけである。

#### product2

In [15]:
%%sql

INSERT INTO product2(name)
VAlUES('モニター');

INSERT INTO product2(name, price)
VALUES('マウス', NULL);

RuntimeError: (pymysql.err.OperationalError) (1364, "Field 'price' doesn't have a default value")
[SQL: INSERT INTO product2(name)
VAlUES('モニター');]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


| テーブル | `price`列の設定 | `NULL`を保存できるか |
|---|---|---|
| `product` | `NOT NULL`指定なし | 可能 |
| `product2` | `price INT NOT NULL` | 不可能 |